In [128]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

In [129]:
IMG_SIZE = 224

def load_image(img_path):
    img = tf.keras.preprocessing.image.load_img(
        img_path, target_size=(IMG_SIZE, IMG_SIZE)
    )
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = img / 255.0
    return img

In [130]:
def generate_triplets(data_dir):
    triplets = []

    persons = os.listdir(data_dir)

    for person in persons:
        person_path = os.path.join(data_dir, person)
        images = os.listdir(person_path)

        if len(images) < 2:
            continue

        for i in range(len(images)-1):
            anchor = os.path.join(person_path, images[i])
            positive = os.path.join(person_path, images[i+1])

            negative_person = random.choice([p for p in persons if p != person])
            negative_path = os.path.join(data_dir, negative_person)
            negative_image = random.choice(os.listdir(negative_path))
            negative = os.path.join(negative_path, negative_image)

            triplets.append((anchor, positive, negative))

    return triplets

In [131]:
def generate_triplet_data(triplets):
    anchor_images = []
    positive_images = []
    negative_images = []

    for a, p, n in triplets:
        anchor_images.append(load_image(a))
        positive_images.append(load_image(p))
        negative_images.append(load_image(n))

    return (
        np.array(anchor_images),
        np.array(positive_images),
        np.array(negative_images)
    )

In [132]:
def build_embedding_model():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False  # first train only top layers

    inputs = layers.Input(shape=(224,224,3))
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128)(x)
    x = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(x)

    return Model(inputs, x)

In [133]:
class TripletModel(tf.keras.Model):

    def __init__(self, embedding_model, margin=0.5):
        super().__init__()
        self.embedding_model = embedding_model
        self.margin = margin

    def train_step(self, data):

        anchor, positive, negative = data

        with tf.GradientTape() as tape:

            emb_anchor = self.embedding_model(anchor, training=True)
            emb_positive = self.embedding_model(positive, training=True)
            emb_negative = self.embedding_model(negative, training=True)

            pos_dist = tf.reduce_sum(tf.square(emb_anchor - emb_positive), axis=1)
            neg_dist = tf.reduce_sum(tf.square(emb_anchor - emb_negative), axis=1)

            loss = tf.maximum(pos_dist - neg_dist + self.margin, 0.0)
            loss = tf.reduce_mean(loss)

        gradients = tape.gradient(loss, self.embedding_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.embedding_model.trainable_variables)
        )

        return {"loss": loss}

In [134]:
triplets = generate_triplets("train")
A, P, N = generate_triplet_data(triplets)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'train'